The trained collaborative filtering models store latent factors for all users as well as all items, which makes the course model file 107 MB. The Streamlit application only needs the item factors, so this notebook extracts them into two small arrays. Nothing is retrained; the saved models are only read.

In [1]:
import pandas as pd
import numpy as np
import pickle

def aligned_factors(cf_model, id_list):
    """Item factors reordered to match my catalogue. Surprise numbers items
    internally during training, so to_inner_iid does the translation."""
    out = np.zeros((len(id_list), cf_model.qi.shape[1]))
    for i, raw_id in enumerate(id_list):
        try:
            out[i] = cf_model.qi[cf_model.trainset.to_inner_iid(raw_id)]
        except ValueError:
            pass
    return out

courses = pd.read_csv('courses_final.csv')
books = pd.read_csv('books_final.csv')

with open('cf_model_courses.pkl', 'rb') as f:
    cf_courses = pickle.load(f)
with open('cf_model_books.pkl', 'rb') as f:
    cf_books = pickle.load(f)

course_factors = aligned_factors(cf_courses, courses['course_id'].tolist())
book_factors = aligned_factors(cf_books, books['book_id'].tolist())

np.save('course_item_factors.npy', course_factors)
np.save('book_item_factors.npy', book_factors)

print("Course factors:", course_factors.shape, " catalogue rows:", len(courses))
print("Book factors:  ", book_factors.shape, " catalogue rows:", len(books))
print("Courses with no training data:", int((course_factors.sum(axis=1) == 0).sum()))

Course factors: (336, 50)  catalogue rows: 336
Book factors:   (593, 50)  catalogue rows: 593
Courses with no training data: 12
